In [4]:
import pathlib
import warnings
import json

import numpy as np
import scipy.sparse as sp
import joblib
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

ROOT      = pathlib.Path("..").resolve()
ARTIFACTS = ROOT / "artifacts"
MODELS    = ROOT / "models"
MODELS.mkdir(exist_ok=True)

RANDOM_STATE  = 42
TEST_SIZE     = 0.20
OPTUNA_TRIALS = 60    # increase for better tuning; 60 is a good balance
N_JOBS        = 1     # set to -1 if RAM > 16 GB, else keep 1 to avoid OOM

print(f"Artifacts : {ARTIFACTS}")
print(f"Models    : {MODELS}")
print(f"Optuna trials : {OPTUNA_TRIALS}")

Artifacts : C:\Users\BIT\APT-Attribution-Engine\artifacts
Models    : C:\Users\BIT\APT-Attribution-Engine\models
Optuna trials : 60


In [5]:
X_sparse      = sp.load_npz(ARTIFACTS / "X_sparse.npz")
label_encoder = joblib.load(ARTIFACTS / "label_encoder.joblib")
feature_vocab = joblib.load(ARTIFACTS / "feature_vocab.joblib")

n_classes = len(label_encoder.classes_)
# y: row index == class index (matrix was built that way in notebook 1)
y = np.arange(n_classes)

print(f"X shape    : {X_sparse.shape}")
print(f"Classes    : {n_classes}")
print(f"Features   : {X_sparse.shape[1]}")


X shape    : (179, 204)
Classes    : 179
Features   : 204


In [22]:
# Groups with only 1 sample can't be stratified — handle gracefully
from collections import Counter

count_by_class = Counter(y)
single_sample_classes = [c for c, n in count_by_class.items() if n < 2]


if single_sample_classes:
    print(f"⚠  {len(single_sample_classes)} classes have only 1 sample — "
          f"they will appear in train only (forced).")



X_train, X_test, y_train, y_test = train_test_split(
    X_sparse, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y, 
    
)

print(f"Train : {X_train.shape[0]} samples")
print(f"Test  : {X_test.shape[0]} samples")


⚠  179 classes have only 1 sample — they will appear in train only (forced).


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [10]:
from imblearn.over_sampling import SMOTE

smote_k = 2   # conservative k — sparse groups have very few samples

class_counts_train = Counter(y_train)
classes_eligible   = [c for c, n in class_counts_train.items() if n >= smote_k + 1]
classes_skipped    = [c for c, n in class_counts_train.items() if n < smote_k + 1]

print(f"Classes eligible for SMOTE : {len(classes_eligible)}")
print(f"Classes skipped (too few)  : {len(classes_skipped)}")

if len(classes_skipped) > 0:
    skipped_names = label_encoder.inverse_transform(classes_skipped[:10])
    print(f"  Sample skipped: {list(skipped_names)}")

try:
    smote = SMOTE(k_neighbors=smote_k, random_state=RANDOM_STATE)
    X_train_res, y_train_res = smote.fit_resample(X_train.toarray(), y_train)
    X_train_res = sp.csr_matrix(X_train_res)
    print(f"\nAfter SMOTE → train size: {X_train_res.shape[0]}")
except Exception as e:
    print(f"SMOTE failed: {e}\nFalling back to original training set.")
    X_train_res, y_train_res = X_train, y_train


Classes eligible for SMOTE : 0
Classes skipped (too few)  : 143
  Sample skipped: [np.str_('ToddyCat'), np.str_('BlackOasis'), np.str_('APT33'), np.str_('Dust Storm'), np.str_('Contagious Interview'), np.str_('LAPSUS$'), np.str_('Magic Hound'), np.str_('Equation'), np.str_('Cleaver'), np.str_('Strider')]

After SMOTE → train size: 143


In [17]:
def make_xgb(params=None):
    defaults = dict(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        verbosity=0,
    )
    if params:
        defaults.update(params)
    return XGBClassifier(**defaults)

def make_rf(params=None):
    defaults = dict(
        n_estimators=300,
        max_depth=None,
        class_weight="balanced_subsample",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    )
    if params:
        defaults.update(params)
    return RandomForestClassifier(**defaults)


def make_svm():
    base = SVC(
        kernel="linear",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=5000,
    )
    return CalibratedClassifierCV(base, cv=3, method="isotonic")


print("Classifier factories defined:")
print("  make_xgb()  — XGBoost (hist tree, mlogloss)")
print("  make_rf()   — Random Forest (balanced_subsample)")
print("  make_svm()  — CalibratedClassifierCV w/ linear SVC (isotonic)")


Classifier factories defined:
  make_xgb()  — XGBoost (hist tree, mlogloss)
  make_rf()   — Random Forest (balanced_subsample)
  make_svm()  — CalibratedClassifierCV w/ linear SVC (isotonic)


In [21]:
print(X_train_res.shape)
print(len(set(y_train_res)))

(143, 204)
143


In [ ]:
X_opt = X_train_res.toarray() if sp.issparse(X_train_res) else X_train_res
cv    = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    xgb_params = dict(
        n_estimators    = trial.suggest_int("xgb_n_est",   100, 500, step=50),
        max_depth       = trial.suggest_int("xgb_depth",   3, 8),
        learning_rate   = trial.suggest_float("xgb_lr",    0.01, 0.3, log=True),
        subsample       = trial.suggest_float("xgb_sub",   0.5, 1.0),
        colsample_bytree= trial.suggest_float("xgb_col",   0.4, 1.0),
    )
    rf_params = dict(
        n_estimators = trial.suggest_int("rf_n_est", 100, 400, step=50),
        max_depth    = trial.suggest_int("rf_depth", 5, 30),
        min_samples_split = trial.suggest_int("rf_min_split", 2, 10),
    )
    w_xgb = trial.suggest_int("w_xgb", 1, 4)
    w_rf  = trial.suggest_int("w_rf",  1, 3)
    w_svm = 1  # SVM weight fixed — too slow to tune here

    ensemble = VotingClassifier(
        estimators=[
            ("xgb", make_xgb(xgb_params)),
            ("rf",  make_rf(rf_params)),
            ("svm", make_svm()),
        ],
        voting="soft",
        weights=[w_xgb, w_rf, w_svm],
        n_jobs=1,
    )

    scores = cross_val_score(
        ensemble, X_opt, y_train_res,
        cv=cv, scoring="f1_macro", n_jobs=1,
    )
    return scores.mean()

print(f"Starting Optuna search — {OPTUNA_TRIALS} trials (this may take 10–30 min)...")
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

print(f"\nBest macro F1 (CV)  : {study.best_value:.4f}")
print(f"Best params         : {study.best_params}")


Starting Optuna search — 60 trials (this may take 10–30 min)...


  0%|          | 0/60 [00:00<?, ?it/s]

[W 2026-06-23 18:52:30,589] Trial 0 failed with parameters: {'xgb_n_est': 250, 'xgb_depth': 8, 'xgb_lr': 0.1205712628744377, 'xgb_sub': 0.7993292420985183, 'xgb_col': 0.4936111842654619, 'rf_n_est': 150, 'rf_depth': 6, 'rf_min_split': 9, 'w_xgb': 3, 'w_rf': 3} because of the following error: ValueError('n_splits=2 cannot be greater than the number of members in each class.').
Traceback (most recent call last):
  File "c:\Users\BIT\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\BIT\AppData\Local\Temp\ipykernel_14688\1069664945.py", line 32, in objective
    scores = cross_val_score(
        ensemble, X_opt, y_train_res,
        cv=cv, scoring="f1_macro", n_jobs=1,
    )
  File "c:\Users\BIT\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\BIT\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 677

ValueError: n_splits=2 cannot be greater than the number of members in each class.